# Notebook 05 — Ensemble Backtest (v3)
Changes from v2:
- Take-profit: 1.5% → **2.5%** (reward:risk now 2.5:1)
- Stop-loss: 2% → **1%**
- Backtest start: 2022-01-01 → **2023-01-01** (true holdout)
- Ensemble confidence threshold: 0.55 → 0.52
- RSI thresholds: 35/65 → 40/60
- Nifty threshold: 1% → 0.3%

In [ ]:
import os
os.chdir(r'C:\Users\Aryan\Desktop\everything\PROJECTS\200%BOT\ai-trading-bot\ai-trading-bot')
print('Working dir:', os.getcwd())

In [ ]:
import os
os.chdir(r'C:\Users\Aryan\Desktop\everything\PROJECTS\200%BOT\ai-trading-bot\ai-trading-bot')
import sys
sys.path.append('.')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data.preprocess import preprocess_symbol, get_feature_columns
from src.models.ensemble import EnsembleModel, TAKE_PROFIT_PCT
from src.strategy.position_sizer import calculate_position_size
print(f'Imports OK | Take-profit: {TAKE_PROFIT_PCT:.1%}')

In [ ]:
ensemble = EnsembleModel(min_votes=2, backtest_mode=True)
ensemble.load_models()
print('Ensemble ready')

## Step 1 — Backtest function

In [ ]:
def run_backtest(symbol, start_date='2023-01-01', capital=100000, phase=1,
                 take_profit_pct=TAKE_PROFIT_PCT, stop_loss_pct=0.01):  # CHANGED: TP=2.5%, SL=1%
    """
    Backtest with improved reward:risk ratio.
    Take-profit: 2.5% | Stop-loss: 1% | Ratio: 2.5:1
    Break-even win rate: 29% (was 57% at 1.5/2.0)
    """
    df = preprocess_symbol(symbol)
    if df is None:
        return None

    bt = df[df.index >= start_date].copy()
    if len(bt) < 60:
        return None

    cap = capital
    trades = []

    for i in range(60, len(bt)):
        row_df = bt.iloc[:i+1]
        today  = bt.index[i]
        curr_close = float(bt['close'].iloc[i])

        sig = ensemble.predict(row_df, symbol)
        if sig.direction == 'NO_TRADE':
            continue
        if i + 1 >= len(bt):
            continue

        entry_price = curr_close
        if sig.direction == 'UP':
            take_profit_price = entry_price * (1 + take_profit_pct)
            stop_loss_price   = entry_price * (1 - stop_loss_pct)
        else:
            take_profit_price = entry_price * (1 - take_profit_pct)
            stop_loss_price   = entry_price * (1 + stop_loss_pct)

        try:
            pos = calculate_position_size(cap, entry_price, stop_loss_price, 1.0, phase)
            quantity = pos['quantity']
        except Exception:
            quantity = max(1, int(cap * 0.02 / (entry_price * stop_loss_pct + 1e-9)))

        if quantity == 0:
            continue

        next_high  = float(bt['high'].iloc[i+1])
        next_low   = float(bt['low'].iloc[i+1])
        next_close = float(bt['close'].iloc[i+1])

        if sig.direction == 'UP':
            if next_high >= take_profit_price:
                exit_price  = take_profit_price
                exit_reason = 'take_profit'
            elif next_low <= stop_loss_price:
                exit_price  = stop_loss_price
                exit_reason = 'stop_loss'
            else:
                exit_price  = next_close
                exit_reason = 'eod'
            pnl = quantity * (exit_price - entry_price)
        else:
            if next_low <= take_profit_price:
                exit_price  = take_profit_price
                exit_reason = 'take_profit'
            elif next_high >= stop_loss_price:
                exit_price  = stop_loss_price
                exit_reason = 'stop_loss'
            else:
                exit_price  = next_close
                exit_reason = 'eod'
            pnl = quantity * (entry_price - exit_price)

        costs   = quantity * entry_price * 0.001
        net_pnl = pnl - costs
        cap    += net_pnl

        trades.append({
            'date':        today,
            'direction':   sig.direction,
            'confidence':  sig.confidence,
            'votes_up':    sig.votes_up,
            'votes_down':  sig.votes_down,
            'entry':       round(entry_price, 2),
            'exit':        round(exit_price, 2),
            'exit_reason': exit_reason,
            'pnl':         round(net_pnl, 2),
            'capital':     round(cap, 2)
        })

    if not trades:
        return None

    df_t   = pd.DataFrame(trades)
    wins   = df_t[df_t['pnl'] > 0]
    losses = df_t[df_t['pnl'] <= 0]
    tp_exits = df_t[df_t['exit_reason'] == 'take_profit']

    return {
        'symbol':        symbol,
        'trades':        len(df_t),
        'win_rate':      round(len(wins) / len(df_t) * 100, 1),
        'return_pct':    round((cap - capital) / capital * 100, 2),
        'final_capital': round(cap, 0),
        'avg_win':       round(wins['pnl'].mean(), 0) if len(wins) else 0,
        'avg_loss':      round(losses['pnl'].mean(), 0) if len(losses) else 0,
        'tp_exits':      len(tp_exits),
        'tp_rate':       round(len(tp_exits) / len(df_t) * 100, 1),
        'trades_df':     df_t
    }

print(f'Backtest function ready | TP={TAKE_PROFIT_PCT:.1%} | SL=1.0% | Ratio=2.5:1')

## Step 2 — Test on top 5 stocks first

In [ ]:
top5 = ['INFY', 'MARUTI', 'ASIANPAINT', 'TATASTEEL', 'SHREECEM']
print('Running backtest on top 5 stocks...')
print(f'Take-profit: {TAKE_PROFIT_PCT:.1%} | Stop-loss: 1.0% | Start: 2023-01-01\n')

results5 = {}
for sym in top5:
    r = run_backtest(sym)
    if r:
        results5[sym] = r
        wl = abs(r['avg_win'] / r['avg_loss']) if r['avg_loss'] != 0 else 0
        print(f"{sym}: {r['trades']} trades | WR={r['win_rate']}% | return={r['return_pct']}% | TP exits={r['tp_rate']}% | W/L={wl:.2f}")

## Step 3 — Run on all 47 stocks

In [ ]:
raw_files = [f.replace('.csv','') for f in os.listdir('data/raw')
             if f.endswith('.csv') and not f.endswith('.NS.csv')]
skip = {'INFRATEL','NIFTY50_all','stock_metadata'}
symbols = [s for s in raw_files if s not in skip]

print(f'Running backtest on {len(symbols)} symbols...')
all_results = []

for sym in symbols:
    r = run_backtest(sym)
    if r:
        all_results.append({k: v for k, v in r.items() if k != 'trades_df'})
        print(f"{sym}: {r['trades']} trades | WR={r['win_rate']}% | return={r['return_pct']}%")

print(f'\nDone: {len(all_results)} backtested')

In [ ]:
res_df = pd.DataFrame(all_results).sort_values('return_pct', ascending=False)

print('Top 10 stocks:')
print(res_df[['symbol','trades','win_rate','return_pct','tp_rate','avg_win','avg_loss']].head(10).to_string(index=False))
print(f'\nMean return: {res_df["return_pct"].mean():.1f}%  (was -44.6% before)')
print(f'Mean win rate: {res_df["win_rate"].mean():.1f}%  (was 48.5% before)')
print(f'Profitable stocks: {(res_df["return_pct"] > 0).sum()}/{len(res_df)}  (was 0/47 before)')

res_df.to_csv('models/results/ensemble_backtest_v3.csv', index=False)
print('\nSaved to models/results/ensemble_backtest_v3.csv')

In [ ]:
# Plot capital curve for best stock
best = res_df.iloc[0]['symbol']
r = run_backtest(best)
df_t = r['trades_df']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

ax1.plot(df_t['date'], df_t['capital'], color='steelblue', lw=2)
ax1.axhline(100000, color='gray', lw=1, linestyle='--', label='Starting capital')
ax1.fill_between(df_t['date'], 100000, df_t['capital'],
                 where=df_t['capital']>=100000, alpha=0.2, color='green', label='Profit')
ax1.fill_between(df_t['date'], 100000, df_t['capital'],
                 where=df_t['capital']<100000, alpha=0.2, color='red', label='Loss')
ax1.set_title(f'Capital Curve — {best} (v3: TP=2.5%, SL=1%)')
ax1.legend()

colors = ['green' if p > 0 else 'red' for p in df_t['pnl']]
ax2.bar(df_t['date'], df_t['pnl'], color=colors, alpha=0.7)
ax2.axhline(0, color='black', lw=0.8)
ax2.set_title('Trade P&L')

plt.tight_layout()
plt.savefig(f'models/results/ensemble_v3_{best}.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved chart for {best}')

## Done
Compare v3 vs v2 results. Key improvements:
- Reward:risk ratio: 0.75 → **2.5**
- Break-even win rate: 57% → **29%**
- Backtest window: 2022-2026 → **2023-2026** (true holdout)